# Retail Data Wrangling and Analytics

In [0]:
%sh
nc -zv 34.26.106.71 5432

In [0]:
%sh
nc -zv 34.26.106.71 5901

In [0]:
%sh
nc -zv google.com 80

In [0]:
# jdbc_url = "jdbc:postgresql://dbserver:5432/dbname"

# df = (
#     spark.read
#     .format("jdbc")
#     .option("url", jdbc_url)
#     .option("dbtable", "schema.tablename")
#     .option("user", "postgres")
#     .option("password", "password")
#     .option("driver", "org.postgresql.Driver")
#     # Performance-critical options
#     # .option("partitionColumn", "id")
#     # .option("lowerBound", "1")
#     # .option("upperBound", "1000000")
#     # .option("numPartitions", "8")
#     .load()
# )

# jdbc_url = "jdbc:postgresql://34.26.106.71:5432/postgres"

# df = (
#     spark.read
#     .format("jdbc")
#     .option("url", jdbc_url)
#     .option("dbtable", "public.retail")
#     .option("user", "postgres")
#     .option("password", "password")
#     .option("driver", "org.postgresql.Driver")
#     # Performance-critical options
#     # .option("partitionColumn", "id")
#     # .option("lowerBound", "1")
#     # .option("upperBound", "1000000")
#     # .option("numPartitions", "8")
#     .load()
# )
# display(df.limit(10))

# df.write.mode("error").saveAsTable("my_table_name") = df.write.saveAsTable("my_table_name")
# df.write.mode("overwrite").saveAsTable("jarvis_training.default.jarvis_spark.retail")

In [0]:
#1. Reading csv to a table : Batch Read

from pyspark.sql.types import *
# from pyspark.sql.functions import current_timestamp, input_file_name, lit

# ==========================================
# 1. Configuration
# ==========================================
# SOURCE_PATH = "/Volumes/catalog/schema/my_volume/raw_data/my_file.csv"
# TARGET_TABLE = "catalog.schema.bronze_user_info"
# WRITE_MODE = "overwrite"  # Common options: overwrite or append

SOURCE_PATH = "/Volumes/jarvis_training/default/retail_analysis/raw/"
TARGET_TABLE = "jarvis_training.default.bronze_table"
WRITE_MODE = "overwrite"  # Common options: overwrite or append
# using append you can run many (e.g. 2) times but data will be double

# ==========================================
# 2. Define Schema (Core for Data Engineering: Strict)
# ==========================================
# Modify the schema based on the actual fields in your CSV
data_schema = StructType([
    StructField("invoice_no", StringType(), True),
    StructField("stock_code", StringType(), True),
    StructField("description", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("invoice_date", TimestampType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("country", StringType(), True)
])

# ==========================================
# 3. Read and Transform (Extraction & Transformation)
# ==========================================
df_raw = (spark.read.format("csv")
          .option("header", "true")   # If your CSV has a header
          #.option("inferSchema", "true") # Infer schema automatically 
          .option("sep", ",")         # Change this if you use a semicolon as delimiter
          .schema(data_schema)        # Force the schema to match the definition
          .load(SOURCE_PATH))

# Add Audit Columns: This is very helpful for troubleshooting in a production environment
# df_final = (df_raw
#             .withColumn("load_timestamp", current_timestamp())  # Record the write timestamp
#             .withColumn("source_file", input_file_name())       # Record the source file name
#             .withColumn("batch_id", lit("20231027_01")))        # Optional: Record the batch ID


# ==========================================
# 4. Write to Delta Table (Loading)
# ==========================================
(df_raw.write
  .format("delta")
  .mode(WRITE_MODE)
  .option("mergeSchema", "true")   # Allow schema evolution if new fields are added in the future
  .saveAsTable(TARGET_TABLE))

print(f"Successfully loaded data to {TARGET_TABLE}")


In [0]:
spark.sql("DROP TABLE IF EXISTS jarvis_training.default.bronze_table")

In [0]:
 %sql
 CREATE VOLUME IF NOT EXISTS retail_analysis

In [0]:
#dbutils.fs.ls("/Volumes/jarvis_training/default/retail_analysis/")
dbutils.fs.mkdirs(
    "/Volumes/jarvis_traning/default/retail_analysis/raw"
)

dbutils.fs.mv(
    "/Volumes/jarvis_training/default/retail_analysis/retail.csv",
    "/Volumes/jarvis_training/default/retail_analysis/raw/retail.csv"
)

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import col, date_format, desc, countDistinct, when, lit, lag, min
from pyspark.sql.functions import date_add, datediff, max as spark_max, concat, lit, sum, countDistinct, desc, ntile, avg
from pyspark.sql.window import Window

In [0]:
#2. Reading csv to a table : Auto Loader
from pyspark.sql.types import *
#from pyspark.sql.functions import current_timestamp, input_file_name

# ==========================================
# 1. Configuration
# ==========================================
# Note: Auto Loader usually monitors a *directory*
# SOURCE_DIR = "/Volumes/catalog/schema/my_volume/raw_data/"
# TARGET_TABLE = "catalog.schema.bronze_user_info_stream"
SOURCE_DIR = "/Volumes/jarvis_training/default/retail_analysis/raw/"
TARGET_TABLE = "jarvis_training.default.bronze_table"

# The checkpoint path is critical; it stores the state of
# which files have already been processed.
# It is recommended to place it in a hidden folder within the same Volume
# or in a dedicated checkpoint directory.
# CHECKPOINT_PATH = "/Volumes/catalog/schema/my_volume/_checkpoints/user_info_stream/"
CHECKPOINT_PATH = "/Volumes/jarvis_training/default/retail_analysis/_checkpoints/retail_info_stream/"

# ==========================================
# 2. Define Schema (Optional but Recommended)
# ==========================================
# Auto Loader supports schema inference, but it is a best practice
# in data engineering to define the schema explicitly.
data_schema = StructType([
    StructField("invoice_no", StringType(), True),
    StructField("stock_code", StringType(), True),
    StructField("description", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("invoice_date", TimestampType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("country", StringType(), True)
])

# ==========================================
# 3. Incremental Read (Auto Loader)
# ==========================================
df_stream = (spark.readStream
  .format("cloudFiles")                 # Required for Auto Loader
  .option("cloudFiles.format", "csv")   # Source file format
  # .option("cloudFiles.schemaLocation", CHECKPOINT_PATH) # Infer schema automatically
  # .option("cloudFiles.inferColumnTypes", "true") # Infer schema automatically, otherwise it will consider all columns as strings
  .option("header", "true")
  .schema(data_schema)                  # Explicit schema definition
  .load(SOURCE_DIR))

# Add audit columns
# df_final = (df_stream
#             .withColumn("load_timestamp", current_timestamp())
#             .withColumn("source_file", input_file_name().alias("source_file")))


# ==========================================
# 4. Write & Trigger
# ==========================================
query = (df_stream.writeStream
  .format("delta")
  .outputMode("append")                 # Append mode for incremental loads
  .option("checkpointLocation", CHECKPOINT_PATH)  # Required for fault tolerance
  .trigger(availableNow=True)           # Key: process all available files once and then stop
  .toTable(TARGET_TABLE))

# Wait for the streaming query to complete before continuing
query.awaitTermination()

print(f"Incremental load to {TARGET_TABLE} completed.")

In [0]:
df = spark.table('jarvis_training.default.bronze_table')
#Persist df in memory for fast futuer access
#df = df.cache()
display(df.count())
print()
display(df.printSchema())

display(df.limit(10))
#df.limit(10).show()

In [0]:
rows = df.count()
cols = len(df.columns)
print(f"Rows: {rows}, Columns: {cols}")

display(df.summary())

# Load CSV into Dataframe
Alternatively, the LGS IT team also dumped the transactional data into a [CSV file](https://raw.githubusercontent.com/jarviscanada/jarvis_data_eng_demo/feature/data/python_data_wrangling/data/online_retail_II.csv). However, the CSV header (column names) doesn't follow the snakecase or camelcase naming convention (e.g. `Customer ID` instead of `customer_id` or `CustomerID`). As a result, you will need to use Pandas to clean up the data before doing any analytics. In addition, unlike the PSQL scheme, CSV files do not have data types associated. Therefore, you will need to cast/convert certain columns into correct data types (e.g. DateTime, numbers, etc..)

**Data Preperation**

- Read the `data/online_retail_II.csv` file into a DataFrame
- Rename all columns to upper camelcase or snakecase
- Convert/cast all columns to the appropriate data types (e.g. datetime)

In [0]:
# 1. Rename Columns (Standardize Column Names)
# In PySpark, you can use withColumnRenamed or select(col().alias())
# renamed_df = df.select(
#     col("Invoice").alias("invoice_no"),
#     col("StockCode").alias("stock_code"),
#     col("Description").alias("description"),
#     col("Quantity").alias("quantity"),
#     col("InvoiceDate").alias("invoice_date"),
#     col("Price").alias("unit_price"),
#     col("Customer ID").alias("customer_id"),  # Fix column name with spaces
#     col("Country").alias("country")
# )

# 2. Type Casting & Missing Value Handling
# Note: PySpark date/time format strings follow Java SimpleDateFormat
# Assume the InvoiceDate format is like "12/1/2009 7:45" (M/d/yyyy H:mm) to yyyy-MM-dd HH:mm:ss (e.g. 2009-12-01 07:45:00)
# cleaned_df = renamed_df \
#     .withColumn("invoice_date", to_timestamp(col("invoice_date"), "M/d/yyyy H:mm")) \
#     .withColumn("unit_price", col("unit_price").cast(DoubleType())) \
#     .withColumn("customer_id", col("customer_id").cast(IntegerType())) \
#     .withColumn("quantity", col("quantity").cast(IntegerType()))

In [0]:
# 3. Filter Invalid Data


# # Drop rows where any column is NULL
# df.dropna(how="any")

# # Drop rows where specific columns are NULL
# df.dropna(subset=[
#     "customer_id",
#     "unit_price"
# ])

# Typically, we need to remove records without a Customer ID
# and records with abnormal values such as unit_price < 0
valid_df = (
    df
    .filter(col("customer_id").isNotNull())
    .filter(col("unit_price") >= 0)
)
display(valid_df.describe())

In [0]:
# 4. Add Derived Columns (Feature Engineering)
# Add total_amount = quantity * unit_price
# Add year_month for easier monthly aggregation later (format: 200912)
final_df = (
    valid_df
    .withColumn("total_amount", col("unit_price") * col("quantity"))
    .withColumn("year_month", date_format(col("invoice_date"), "yyyyMM").cast(IntegerType()))            
)

print("Schema after data cleaning:")
final_df.printSchema()

print("Preview of cleaned data:")
display(final_df.limit(10))

# Total Invoice Amount Distribution

---
**Please remove this insturction cell after you are done with coding**

1. Calculate the invoice amount. Note: an invoice consists of one or more items where each item is a row in the df. (hint: you need to `GROUP BY invoice`)
2. Draw the distribution of invoice amount with min, max, median, mod, and mean. However, you will notice many outlier data (e.g. invoices with large amounts). Sample hist and box charts:

![](https://i.imgur.com/N8hsbDa.jpg)

3. Draw the distribution for the first 85 quantiles of the invoice amount data with min, max, median, mod, and mean.


![](https://i.imgur.com/tJrH1qj.jpg)


---

In [0]:
from pyspark.sql.functions import sum
# --- Task: Calculate the total amount for each invoice ---
invoice_df = (
    final_df
    .groupBy("invoice_no")
    .agg(sum("total_amount").alias("invoice_total"))
)
# 1. View summary statistics (Min, Max, Mean, Stddev, etc.)
# Equivalent to Pandas' .describe()
print("Invoice total amount statistics:")
invoice_df.select("invoice_total").summary().show()

# 2. Visualize the distribution
# After running, click the "+" button below the table -> Visualization -> Histogram & Box plot
# Select 'invoice_total' for the X-axis for hist & 'invoice_total' for the y-axis for box plot
display(invoice_df.limit(10))

In [0]:
# --- Task: Invoice Amount Distribution (Top 85% Quantile) ---

# 1. Calculate the 85th percentile threshold
# Parameters: "column name", [list of quantiles], relative error
quantiles = invoice_df.stat.approxQuantile("invoice_total", [0.85], 0.01)
quantile_85_val = quantiles[0]

print(f"The 85th percentile threshold amount is: {quantile_85_val}")

# 2. Filter the data: keep only invoices with amounts less than or equal to the threshold
# and typically focus on positive invoices (> 0), filtering out refunds and near-zero values
filtered_invoice_df = invoice_df.filter(
    (col("invoice_total") <= quantile_85_val) & 
    (col("invoice_total") > 0)
)

# 3. Calculate summary statistics (Min, Max, Mean, Median)
# summary() automatically computes count, mean, stddev, min,
# 25%, 50% (Median), 75%, and max
print("Summary statistics for the top 85% of the data:")
filtered_invoice_df.select("invoice_total").summary().show()

In [0]:
from pyspark.sql.functions import desc

# 4. Calculate the mode
# Spark does not have a built-in mode function, so we compute it via grouping
mode_row = filtered_invoice_df.groupBy("invoice_total") \
    .count() \
    .orderBy(desc("count")) \
    .first()

display(mode_row)
print()
print(f"Mode: {mode_row['invoice_total']} (appears {mode_row['count']} times)")

# 5. Visualize the distribution
# After running, click "+" -> Visualization -> Histogram (X-axis: invoice_total)
display(filtered_invoice_df.limit(10))

# Monthly Placed and Canceled Orders

---
**Please remove this insturction cell after you are done with coding**

- The attribute information (see the `project kick-off` section) contains useful information that helps you to identify canceled orders
- To simplify the problem, you can assume that there are two invoice numbers for each canceled order (one for the original invoice and one for the canceled invoice). Therefore, `# of placed orders = total # of orders - 2 * canceled order`. Furthermore, you can also assume the original invoice and canceled invoice are on always on the same day (this eliminate the case where the original invoice and canceled invoices are on different months)
- hints: you might want to create a new integer column with YYYYMM format. e.g. `2009-12-01 07:45:00 -> 200912` which allows easy GROUP BY.

**Sample Plot:**

![](https://i.imgur.com/tmLsPDf.jpg)

---

In [0]:
# --- Task：Monthly Placed and Canceled Orders ---

# 1. Mark whether an order is canceled (Is Cancelled?)
# Assume that InvoiceNo starting with "C" indicates a canceled order
# We are using the original cleaned final_df for this task, as the invoice_df has already been aggregated and lost the 'C' information

from pyspark.sql.functions import countDistinct, when, col, lit

marked_df = (
    final_df
    .withColumn("is_canceled", when(col("invoice_no").startswith("C"),1).otherwise(0))
)

# 2. Group by month and count: Total Orders & Canceled Orders
# Note: We are counting "invoice numbers", so we use countDistinct("invoice_no")

monthly_orders_stats = (
    marked_df
    .groupBy("year_month")
    .agg(
        countDistinct("invoice_no").alias("total_orders"),
        # Count the number of canceled orders: filter out canceled orders and then countDistinct
        countDistinct(when(col("is_canceled") == 1, col("invoice_no"))).alias("canceled_orders"))
)

result_orders_df = (
    monthly_orders_stats
    .withColumn("placed_orders",
    col("total_orders") - (lit(2) * col("canceled_orders")))
    .orderBy("year_month")
)

# 4. Display the result
# Visualization: Line Chart or Bar Chart
# X: year_month
# Y: Add two columns -> placed_orders and canceled_orders
print("Monthly Order Statistics (Placed vs Canceled):")
display(result_orders_df)

# Monthly Sales

---
**Please remove this insturction cell after you are done with coding**


- Calculate the monthly sales data
- Plot a chart to show monthly sales (e.g. x-asix=year_month, y-axis=sales_amount)

![](https://i.imgur.com/k1KOqKO.jpg)

---

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import sum, lag, col
# 1. Calculate Monthly Sales Total
monthly_sales_df = (
    final_df
    .groupBy("year_month")
    .agg(sum("total_amount").alias("monthly_sales"))
)

# Display the result
# Chart: Line Chart (X-axis: year_month, Y-axis: growth_percent)
display(monthly_sales_df)

# Monthly Sales Growth


---
**Please remove this insturction cell after you are done with coding**

- Calculate monthly sales percentage growth data
- Plot a chart to show the growth percentage

![](https://i.imgur.com/J3btp8j.jpg)

---

In [0]:

# 2. Calculate Month-over-Month Growth Rate (Growth Rate)
# Define window: sort by month
window_spec = Window.orderBy("year_month")

# Use lag function to get "previous month's" sales
monthly_growth_df = (
    monthly_sales_df
    .withColumn("prev_month_sales", lag("monthly_sales").over(window_spec)) 
    .withColumn("growth_percent", 
                (col("monthly_sales") - col("prev_month_sales")) / col("prev_month_sales") * 100)
    .orderBy("year_month")
)

# Display the result
# Suggested visualizations:
# Chart 1: Bar Chart (X-axis: year_month, Y-axis: monthly_sales)
# Chart 2: Line Chart (X-axis: year_month, Y-axis: growth_percent)
display(monthly_growth_df)


# Monthly Active Users

---
**Please remove this insturction cell after you are done with coding**

- Compute # of active users (e.g. unique `CusotomerID`) for each month
- Plot a bar chart

![](https://i.imgur.com/eFYp8VF.jpg)

---

In [0]:
# Calculate MAU (Monthly Active Users)
mau_df = (
    final_df.groupBy("year_month")
    .agg(countDistinct("customer_id").alias("active_users"))
    .orderBy("year_month")
)
# Visualization: Bar Chart (X: year_month, Y: active_users)
display(mau_df)

# New and Existing Users



---
**Please remove this insturction cell after you are done with coding**

- Plot a diagram to show new and exiting user for each month.
- A user is identified as a new user when he/she makes the first purchase
- A user is identified as an existing user when he/she made purchases in the past
- hints:
  - find out the first purchase year-month for each user and then join this data with the transactional data to help you identified new/exiting users

![](https://i.imgur.com/nWjnrpr.jpg)

---

In [0]:
# from pyspark.sql.functions import min
# user_first_purchase_df = (
#     final_df.groupby("customer_id")
#     .agg(min("year_month").alias("first_purchase_month"))
# )

# joined_df = final_df.select("customer_id", "year_month").distinct() \
#     .join(user_first_purchase_df, on="customer_id", how="left")

# user_type_df = joined_df.withColumn("user_type", 
#     when(col("year_month") == col("first_purchase_month"), "New")
#     .otherwise("Existing")
# )

In [0]:
display(joined_df.limit(10))

In [0]:
display(user_type_df.limit(10))

In [0]:
from pyspark.sql.functions import count, when
# Step 1: Find each user's "First Purchase Month" (Cohort Month)
user_first_purchase_df = final_df.groupBy("customer_id") \
    .agg(min("year_month").alias("first_purchase_month"))

# Step 2: Join the First Purchase Month back to the main data
# This way, each transaction will know when the user first made a purchase
joined_df = final_df.select("customer_id", "year_month").distinct() \
    .join(user_first_purchase_df, on="customer_id", how="left")

# Step 3: Label as New or Existing
# If the transaction month is the same as the first purchase month, label as "New"
user_type_df = joined_df.withColumn("user_type", 
    when(col("year_month") == col("first_purchase_month"), "New")
    .otherwise("Existing")
)

# Step 4: Calculate the count of New and Existing users by month
new_existing_stats = user_type_df.groupBy("year_month", "user_type") \
    .agg(count("customer_id").alias("user_count")) \
    .orderBy("year_month", "user_type")

# Visualization Type: Bar Chart
# X Column: year_month
# Y Column: user_count
# Group/Color By: user_type (This step is key for creating a stacked bar chart)
# Stacking: Stack
display(new_existing_stats)

## Finding RFM

RFM is a method used for analyzing customer value. It is commonly used in database marketing and direct marketing and has received particular attention in the retail and professional services industries. ([wikipedia](https://en.wikipedia.org/wiki/RFM_(market_research)))

Optional Reading: [Making Your Database Pay Off Using Recency Frequency and Monetary Analysis](http://www.dbmarketing.com/2010/03/making-your-database-pay-off-using-recency-frequency-and-monetary-analysis/)


RFM stands for three dimensions:

- Recency – How recently did the customer purchase?

- Frequency – How often do they purchase?

- Monetary Value – How much do they spend?

Note: To simplify the problem, let's keep all placed and canceled orders.


**Sample RFM table**

![](https://i.imgur.com/sXFIg6u.jpg)

In [0]:
from pyspark.sql.functions import date_add, datediff, max as spark_max, concat, lit, sum, countDistinct, desc, ntile, avg
from pyspark.sql.window import Window

In [0]:
# 1. Determine the "Reference Date"
# Logic: The maximum date in the dataset + 1 day
# Note: We use collect() to get a scalar value instead of keeping the DataFrame format
max_date_row = final_df.agg(spark_max("invoice_date")).collect()[0][0]
print(f"Last date in the dataset: {max_date_row}")

rfm_df = (
    final_df
    .groupBy("customer_id")
    .agg(
        datediff(date_add(lit(max_date_row), 1), spark_max("invoice_date")).alias("recency"),
        countDistinct("invoice_no").alias("frequency"),
        sum("total_amount").alias("monetary")
    )
)
# 3. Filtering
# Equivalent to Pandas: rfm_df = rfm_df.filter(col("monetary") > 0)
rfm_df = rfm_df.filter(rfm_df.monetary >0)

display(rfm_df.limit(10))

In [0]:
display(final_df.agg(spark_max("invoice_date")))

# RFM Segmentation

---
**Please remove this insturction cell after you are done with coding**
RFM segmentation categorizes your customers into different segments, according to their interactions with your website, which will allow you to subsequently approach these groups in the most effective way. In this article, we will show you how to make an RFM segmentation based on an RFM score combining all three RFM parameters together and allowing you to divide your customers into 11 different segments. 

- [RFM Segmentation business cases](https://docs.exponea.com/docs/rfm-segmentation-business-use)

- [RFM Segmentation Guide](https://docs.exponea.com/docs/rfm-segmentation-business-use)

As you can see, computing RFM segmentation requires extensive domain knowledge in marketing which is out of the scope in this project. In practice, you will work with BA/DA to figure out how to compute RFM segments. To simplify this project, a [sample RFM segmentation Notebook](https://github.com/jarviscanada/jarvis_data_eng_demo/blob/feature/data/python_data_wrangling/ipynb/customer-segmentation-with-rfm-score.ipynb) is provided. You are responsible to understand everything from that Notebook and then integrate it into yours. 

- Download the [sample notebook](https://github.com/jarviscanada/jarvis_data_eng_demo/blob/feature/data/python_data_wrangling/ipynb/customer-segmentation-with-rfm-score.ipynb) and import to your Jupyter Notebook or VSCode
- Run the notebook and understand all cells
- Read the remark section at the end of the notebook. You will need this information when writing the README file
- Integrate the RFM segmentation calculation into your notebook

---

In [0]:
# Define window specifications
# Recency: Sort in descending order (higher value = longer since last purchase = lower score, lower value = recent purchase = higher score)
r_window = Window.orderBy(desc("recency"))
# Frequency & Monetary: Sort in ascending order (higher value = higher score)
f_window = Window.orderBy("frequency")
m_window = Window.orderBy("monetary")

# Calculate 1-5 scores (NTILE)
rfm_scored_df = rfm_df \
    .withColumn("r_score", ntile(5).over(r_window)) \
    .withColumn("f_score", ntile(5).over(f_window)) \
    .withColumn("m_score", ntile(5).over(m_window))

# Construct RFM score combination string (e.g., "55")
# Only R and F scores are needed for lookup
rfm_final_df = rfm_scored_df.withColumn("rf_score", 
    concat(col("r_score").cast("string"), col("f_score").cast("string"))
)

display(rfm_final_df.limit(5))

In [0]:
# Use rlike (Regex Like) for pattern matching
segmented_df = rfm_final_df.withColumn("segment", 
    when(col("rf_score").rlike("^[1-2][1-2]$"), "Hibernating")
    .when(col("rf_score").rlike("^[1-2][3-4]$"), "At Risk")
    .when(col("rf_score").rlike("^[1-2]5$"), "Can't Lose")
    .when(col("rf_score").rlike("^3[1-2]$"), "About to Sleep")
    .when(col("rf_score").rlike("^33$"), "Need Attention")
    .when(col("rf_score").rlike("^[3-4][4-5]$"), "Loyal Customers")
    .when(col("rf_score").rlike("^41$"), "Promising")
    .when(col("rf_score").rlike("^51$"), "New Customers")
    .when(col("rf_score").rlike("^[4-5][2-3]$"), "Potential Loyalists")
    .when(col("rf_score").rlike("^5[4-5]$"), "Champions")
    .otherwise("Other")  # Default case for unmatched segments
)

print("RFM Segmentation Preview:")
display(segmented_df.select("customer_id", "recency", "frequency", "monetary", "rf_score", "segment").limit(10))


In [0]:
# 1. Calculate the mean metrics for each segment
segment_stats = segmented_df.groupBy("segment") \
    .agg(
        count("customer_id").alias("count"),
        avg("recency").alias("avg_recency"),
        avg("frequency").alias("avg_frequency"),
        avg("monetary").alias("avg_monetary")
    ) \
    .orderBy(desc("count"))

print("Detailed Metrics for Each Segment:")
display(segment_stats)

# 2. Visualize the distribution (Treemap or Bar Chart)
# Click the "+" on the result below -> Visualization
# Recommended:
# Type: Bar Chart
# X Column: segment
# Y Column: count